[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philippe-amuzuga/Meca-PY_Ed2026/blob/main/python/P10/P10-1_Calcul_Symbolique_CORRIGE.ipynb)

# Calcul symbolique avec SymPy

Chapitre 10 du livre *Python pour l'ingénierie mécanique, industrielle et productique*.

## Cadrage

**Contexte industriel.** Votre bureau d'études écrit la flèche d'une poutre, la dérive pour savoir quelle dimension compte le plus, l'inverse pour trouver la longueur qui tient une flèche admissible, puis recopie le résultat dans un programme. À chaque recopie, une puissance, un signe ou une conversion peut se glisser dans la formule. Vous voudriez que l'ordinateur manipule la formule elle-même, en lettres, et ne la change en nombres qu'à la fin, avec une valeur de référence exacte pour contrôler vos calculs ?

**Objectif terminal.** Ce chapitre montre comment, avec le calcul symbolique de la bibliothèque SymPy.

**Objectif instrumental.**

- Écrire une formule avec des symboles, la simplifier, la dériver, l'intégrer.
- Résoudre en lettres une équation, un système et une équation différentielle.
- Écrire une équation caractéristique avec une matrice littérale.
- Transformer une expression en fonction NumPy et la confronter à une valeur exacte.

**Critère de résultat.** Chaque exercice atteint son résultat attendu, que des contrôles du notebook vérifient.

**Vérification.** Beaucoup d'exercices donnent une seconde voie : un calcul fait par deux chemins indépendants est un calcul qu'on peut croire.

**Prérequis.** Les chapitres 3 à 6, après une filière scientifique.

## Source de données

Aucun jeu de données : les valeurs viennent de l'énoncé ou sont engendrées par le code, à graine fixée.

## Exercices

| N° | Exercice | Durée indicative |
|:-|:-|-:|
| 10.1 | La flèche d'une poutre en lettres | 15 min |
| 10.2 | Arbre en torsion et deux masses | 20 min |
| 10.3 | Fréquences propres et treillis (défi) | 30 min |

Dans le sujet, les cellules marquées `# Votre code :` sont à écrire ; le corrigé les donne résolues. Chaque exercice se termine par des contrôles qui comparent votre résultat à celui du livre.

### En-tête de reproductibilité

Versions des bibliothèques employées : les valeurs du livre sont celles de ces versions, avec des graines fixées.

In [1]:
# En-tête de reproductibilité : versions employées par ce notebook.
import platform
from importlib.metadata import version

print("Python", platform.python_version())
print("numpy", version("numpy"))
print("sympy", version("sympy"))

Python 3.12.0
numpy 2.4.6
sympy 1.14.0


### Préparation du dépôt (Colab)

Ouvert seul dans Colab, ce notebook ne voit pas les dossiers du dépôt. La cellule suivante ne télécharge rien : elle cherche `donnees/` et `outils/` (à deux dossiers au-dessus du notebook, à côté de lui ou dans un clone nommé `Meca-PY_Ed2026`), ajoute `outils/` au chemin de recherche de Python, et, sur Colab, liste ce qui manque. Pour le fournir, ouvrez le panneau des fichiers de Colab, créez les dossiers `donnees` et `outils` à la racine de l'espace de travail, et téléversez-y les fichiers listés, pris dans le dépôt. Ce notebook n'a besoin d'aucun jeu ni d'aucun module : la cellule ne signale rien, mais elle prépare l'import des modules du livre dans vos propres cellules.

In [2]:
# Préparation du dépôt (utile sur Colab) : cette cellule ne télécharge rien.
import os
import sys
from pathlib import Path

JEUX_LUS = []
MODULES_DU_LIVRE = []                          # fichiers de outils/


def chercher(dossier):
    """Le dossier du dépôt cherché depuis le notebook, puis depuis un clone ; None s'il est introuvable."""
    for base in (Path("..", ".."), Path("."), Path("Meca-PY_Ed2026")):
        if (base / dossier).is_dir():
            return base / dossier
    return None


donnees, outils = chercher("donnees"), chercher("outils")
if outils is not None and str(outils) not in sys.path:
    sys.path.insert(0, str(outils))
manque = [f"donnees/{nom}" for nom in JEUX_LUS if donnees is None or not (donnees / nom).is_file()]
manque += [f"outils/{nom}" for nom in MODULES_DU_LIVRE if outils is None or not (outils / nom).is_file()]
if manque and "COLAB_RELEASE_TAG" in os.environ:          # variable posée par Colab
    print("À téléverser dans l'espace de travail de Colab, panneau des fichiers :")
    print("\n".join("  " + nom for nom in manque))


### Préparation

In [3]:
import math
import numpy as np
import sympy as sp

## Exercice 10.1 : La flèche d'une poutre en lettres (15 min)

Écrivez en lettres la flèche d'une poutre sur deux appuis chargée en son milieu, $f = F L^3/(48\,E\,I)$, avec $I = b\,h^3/12$, et simplifiez-la. Remplacez les lettres par les valeurs de l'exercice du chapitre 3 ($F = 2\,000$ N, $L = 1\,000$ mm, $E = 210\,000$ MPa, $b = 20$ mm, $h = 40$ mm), gardez la valeur exacte, puis chiffrez-la. Dérivez enfin $f$ par rapport à $h$ et calculez la variation relative de la flèche pour une variation relative de $h$.

**Résultat attendu.** Flèche simplifiée $f = \frac{F L^{3}}{4 E b h^{3}}$ ; valeur exacte $\frac{625}{336}$ mm, soit 1,8601 mm ; dérivée $- \frac{3 F L^{3}}{4 E b h^{4}}$ ; la flèche varie comme $h^{-3}$.

**Seconde voie.** La fonction du chapitre 3 (section 3.2) redonne 1,86 mm, et le rapport des flèches de $30$ et de $60$ mm vaut huit, soit $2^3$.

**Indication.** Déclarez les lettres positives, posez le moment quadratique $b\,h^3/12$ et simplifiez la flèche. La méthode `subs`, avec un dictionnaire, donne la fraction exacte, `evalf` sa valeur décimale. La variation relative est $h\,f'/f$, que `simplify` réduit à $-3$.

In [4]:
# Solution :
F, L, E, b, h = sp.symbols("F L E b h", positive=True)
I = sp.Symbol("I", positive=True)                  # moment quadratique, laissé en lettre
fleche_brute = F * L**3 / (48 * E * I)
print("flèche brute      :", fleche_brute)
I_rect = b * h**3 / 12                              # section rectangulaire b x h
fleche = sp.simplify(fleche_brute.subs(I, I_rect))
print("flèche simplifiée :", fleche)
cas = {F: 2000, L: 1000, E: 210000, b: 20, h: 40}   # N, mm, MPa, mm, mm : le cas du chapitre 3
fleche_exacte = fleche.subs(cas)
fleche_mm = float(fleche_exacte)
print("flèche exacte (fraction) :", fleche_exacte, "  décimale :", fleche_exacte.evalf(6), "mm")
# Fin de la solution
assert abs(fleche_mm - 1.8601) < 5e-5, "flèche attendue : 1,8601 mm"
# Solution :
fleche_flottants = 2000.0 * 1000.0**3 / (48 * 210000.0 * (20.0 * 40.0**3 / 12))      # formule du chapitre 3
# Fin de la solution
assert abs(fleche_mm - fleche_flottants) < 1e-12, "formule du chapitre 3"
# Solution :
# Dériver : de combien varie la flèche quand la hauteur h change ?
dfleche_dh = sp.diff(fleche, h)
sensibilite = sp.simplify(h * dfleche_dh / fleche)     # variation relative de f pour une variation relative de h
dfleche_dh_num = float(dfleche_dh.subs(cas))
print("dérivée par rapport à h :", dfleche_dh)
print("sensibilité relative    :", sensibilite, "  (la flèche varie comme h**(-3))")
print(f"à h = 40 mm : {dfleche_dh_num:.4f} mm de flèche par mm de hauteur")
# Fin de la solution
assert sensibilite == -3, "la flèche varie comme h à la puissance moins trois"
assert abs(dfleche_dh_num - (-3 * fleche_mm / 40)) < 1e-12
# Solution :
# Intégrer : l'équation E I y'' = M(x) donne la ligne élastique y(x) sans la connaître d'avance (y vers le haut).
# Moment fléchissant de la demi-poutre gauche, M(x) = F x / 2 ; y(0) = 0 (appui) et y'(L/2) = 0 (symétrie).
x, C1, C2 = sp.symbols("x C1 C2")
moment = F * x / 2
EI_y1 = sp.integrate(moment, x) + C1               # E I y'
EI_y = sp.integrate(EI_y1, x) + C2                 # E I y
constantes = sp.solve([EI_y1.subs(x, L / 2), EI_y.subs(x, 0)], [C1, C2])
y = sp.simplify((EI_y / (E * I)).subs(constantes))
y_milieu = sp.simplify(y.subs(x, L / 2))
print("ligne élastique y(x)    :", sp.factor(y))
print("flèche au milieu -y(L/2) :", -y_milieu)
# Fin de la solution
assert sp.simplify(-y_milieu - fleche_brute) == 0, "l'intégration doit redonner F L**3 / (48 E I)"
# Solution :
# Développer et factoriser : Kt de Heywood (plaque percée en traction), moment quadratique d'un tube
r = sp.Symbol("r", positive=True)                  # r = d / W
kt_heywood = 2 + (1 - r)**3
kt_developpe = sp.expand(kt_heywood)
print("Kt de Heywood développé :", kt_developpe, "  Kt(0) =", kt_developpe.subs(r, 0))
D, d_int = sp.symbols("D d_int", positive=True)
I_tube = sp.pi * (D**4 - d_int**4) / 64
I_tube_facteurs = sp.factor(I_tube)
print("moment quadratique d'un tube, factorisé :", I_tube_facteurs)
# Fin de la solution
assert sp.expand(kt_developpe - kt_heywood) == 0 and kt_developpe.subs(r, 0) == 3
assert sp.expand(I_tube_facteurs - I_tube) == 0
print("Exercice 10.1 : contrôles passés")

flèche brute      : F*L**3/(48*E*I)
flèche simplifiée : F*L**3/(4*E*b*h**3)
flèche exacte (fraction) : 625/336   décimale : 1.86012 mm
dérivée par rapport à h : -3*F*L**3/(4*E*b*h**4)
sensibilité relative    : -3   (la flèche varie comme h**(-3))
à h = 40 mm : -0.1395 mm de flèche par mm de hauteur
ligne élastique y(x)    : F*x*(-3*L**2 + 4*x**2)/(48*E*I)
flèche au milieu -y(L/2) : F*L**3/(48*E*I)
Kt de Heywood développé : -r**3 + 3*r**2 - 3*r + 3   Kt(0) = 3
moment quadratique d'un tube, factorisé : pi*(D - d_int)*(D + d_int)*(D**2 + d_int**2)/64
Exercice 10.1 : contrôles passés


## Exercice 10.2 : Arbre en torsion et deux masses (20 min)

Un arbre plein transmet un couple de $500$ N$\cdot$m avec une contrainte de cisaillement admissible de $60$ MPa ; la contrainte vaut $\tau = 16\,T/(\pi\,d^3)$. Résolvez en lettres pour $d$, puis chiffrez le diamètre, avec le couple en N$\cdot$mm. Deux masses $m_1$ et $m_2$ sont reliées entre elles par un ressort $k_2$, et la première au bâti par un ressort $k_1$ : leurs matrices sont $M = \mathrm{diag}(m_1, m_2)$ et $K = \begin{pmatrix} k_1 + k_2 & -k_2 \\ -k_2 & k_2 \end{pmatrix}$. Écrivez l'équation caractéristique $\det(K - \lambda M) = 0$, avec $\lambda = \omega^2$, puis les deux fréquences propres $f = \sqrt{\lambda}/(2\pi)$ pour $m_1 = 5$ kg, $m_2 = 3$ kg, $k_1 = 2$ MN/m et $k_2 = 1$ MN/m.

**Résultat attendu.** Diamètre de 34,88 mm ; fréquences de 66,78 Hz et 138,5 Hz ; équation caractéristique $k_{1} k_{2} + \lambda^{2} m_{1} m_{2} + \lambda \left(- k_{1} m_{2} - k_{2} m_{1} - k_{2} m_{2}\right)= 0$.

**Seconde voie.** La formule fermée $d = \bigl(16\,T/(\pi\,\tau)\bigr)^{1/3}$ redonne 34,8816 mm, et la formule du second degré appliquée aux trois coefficients du polynôme redonne les deux racines en $\lambda$.

**Indication.** Résolvez `Eq(16*T/(pi*d**3), tau)` pour `d` avec des symboles positifs, puis `subs` avec $T = 500\,000$ N$\cdot$mm. Pour les masses, `Matrix` et `diag` donnent $K$ et $M$ ; le déterminant de $K - \lambda M$, développé, est le polynôme demandé. Ses racines en $\lambda$ sont à remplacer par les valeurs numériques, puis à passer à la racine carrée et à diviser par $2\pi$.

In [5]:
# Solution :
# Arbre plein en torsion : tau = 16 T / (pi d**3). Quel diamètre pour T et tau donnés ?
T, tau, d = sp.symbols("T tau d", positive=True)
equation_arbre = sp.Eq(16 * T / (sp.pi * d**3), tau)
solutions = sp.solve(equation_arbre, d)
d_litteral = solutions[0]
print("équation :", equation_arbre, "  nombre de solutions réelles positives :", len(solutions))
print("diamètre en lettres :", d_litteral)
T_Nmm, tau_adm_MPa = 500 * 1000, 60                  # 500 N.m exprimés en N.mm, 60 MPa
d_arbre_mm = float(d_litteral.subs({T: T_Nmm, tau: tau_adm_MPa}))
print(f"T = 500 N.m, tau = 60 MPa : d = {d_arbre_mm:.4f} mm")
# Fin de la solution
assert sp.simplify(d_litteral**3 - 16 * T / (sp.pi * tau)) == 0
assert abs(d_arbre_mm - (16 * T_Nmm / (math.pi * tau_adm_MPa))**(1 / 3)) < 1e-12
assert abs(d_arbre_mm - 34.88) < 0.005, "diamètre attendu : 34,88 mm"
# Solution :
# Un système linéaire de deux équations : réactions d'appui d'une poutre chargée à la distance a de l'appui A.
RA, RB, a = sp.symbols("R_A R_B a", positive=True)
reactions = sp.linsolve([sp.Eq(RA + RB, F), sp.Eq(RB * L, F * a)], [RA, RB])
RA_litterale, RB_litterale = list(reactions)[0]            # linsolve rend un ensemble de solutions
RA_litterale, RB_litterale = sp.simplify(RA_litterale), sp.simplify(RB_litterale)
print("réactions d'appui :", "R_A =", RA_litterale, " R_B =", RB_litterale)
# Fin de la solution
assert sp.simplify(RA_litterale - F * (L - a) / L) == 0
assert sp.simplify(RB_litterale - F * a / L) == 0
assert RA_litterale.subs({F: 2000, L: 1000, a: 500}) == 1000
# Solution :
# Deux masses reliées par deux ressorts : m1, m2 (kg), k1, k2 (N/m).
# Équation caractéristique det(K - omega**2 M) = 0, écrite avec lam = omega**2.
m1, m2, k1, k2 = sp.symbols("m1 m2 k1 k2", positive=True)
lam = sp.Symbol("lambda", positive=True)           # lam = omega au carré
M_masses = sp.diag(m1, m2)
K_raideur = sp.Matrix([[k1 + k2, -k2], [-k2, k2]])
polynome = sp.expand((K_raideur - lam * M_masses).det())
polynome_attendu = m1 * m2 * lam**2 - (k1 * m2 + k2 * m1 + k2 * m2) * lam + k1 * k2
print("équation caractéristique :", sp.collect(polynome, lam), "= 0")
# Fin de la solution
assert sp.expand(polynome - polynome_attendu) == 0, "équation à deux degrés de liberté"
# Solution :
modes_cas = {m1: 5, m2: 3, k1: 2000000, k2: 1000000}     # kg, kg, N/m, N/m
racines = sp.solve(polynome.subs(modes_cas), lam)
frequences_Hz = sorted(float(sp.sqrt(rac) / (2 * sp.pi)) for rac in racines)
print("racines en lambda :", racines)
print(f"fréquences propres : {frequences_Hz[0]:.2f} Hz et {frequences_Hz[1]:.2f} Hz")
# Fin de la solution
assert len(racines) == 2 and abs(frequences_Hz[0] - 66.78) < 0.005
assert abs(frequences_Hz[1] - 138.5) < 0.05
# Solution :
# Équation différentielle : oscillateur sous-amorti x'' + 2 alpha x' + (alpha**2 + wd**2) x = 0,
# écarté de x0 puis lâché sans vitesse.
t = sp.Symbol("t", positive=True)
x_f = sp.Function("x")
alpha, wd, x0 = sp.symbols("alpha omega_d x_0", positive=True)
edo = sp.Eq(x_f(t).diff(t, 2) + 2 * alpha * x_f(t).diff(t) + (alpha**2 + wd**2) * x_f(t), 0)
solution_edo = sp.dsolve(edo, x_f(t), ics={x_f(0): x0, x_f(t).diff(t).subs(t, 0): 0})
x_exact = sp.simplify(solution_edo.rhs)
print("solution de l'équation différentielle :", x_exact)
# Décrément logarithmique : rapport de deux maxima successifs, séparés de la pseudo-période 2 pi / wd.
t_pic = sp.Symbol("t_pic", positive=True)
rapport_maxima = sp.simplify(sp.expand(x_exact.subs(t, t_pic) / x_exact.subs(t, t_pic + 2 * sp.pi / wd),
                                       trig=True))
decrement_litteral = sp.simplify(sp.log(rapport_maxima))
print("rapport de deux maxima successifs :", rapport_maxima)
print("décrément logarithmique en lettres :", decrement_litteral)
# Fin de la solution
assert sp.simplify(decrement_litteral - 2 * sp.pi * alpha / wd) == 0, "2 pi alpha / omega_d"
# Solution :
masse_kg, raideur_Nm, zeta = 5.0, 20000.0, 0.05       # oscillateur du chapitre d'intégration
omega0 = math.sqrt(raideur_Nm / masse_kg)
alpha_val = zeta * omega0
wd_val = omega0 * math.sqrt(1 - zeta**2)
decrement = float(decrement_litteral.subs({alpha: alpha_val, wd: wd_val}))
f0_Hz = omega0 / (2 * math.pi)
print(f"m = 5 kg, k = 20 000 N/m, zeta = 0,05 : f0 = {f0_Hz:.2f} Hz, décrément logarithmique = {decrement:.4f}")
# Fin de la solution
assert abs(decrement - 2 * math.pi * zeta / math.sqrt(1 - zeta**2)) < 1e-12
assert abs(decrement - 0.3146) < 5e-5 and abs(f0_Hz - 10.07) < 0.005
print("Exercice 10.2 : contrôles passés")

équation : Eq(16*T/(pi*d**3), tau)   nombre de solutions réelles positives : 1
diamètre en lettres : 2*2**(1/3)*T**(1/3)/(pi**(1/3)*tau**(1/3))
T = 500 N.m, tau = 60 MPa : d = 34.8816 mm
réactions d'appui : R_A = F*(L - a)/L  R_B = F*a/L
équation caractéristique : k1*k2 + lambda**2*m1*m2 + lambda*(-k1*m2 - k2*m1 - k2*m2) = 0
racines en lambda : [1400000/3 - 200000*sqrt(19)/3, 200000*sqrt(19)/3 + 1400000/3]
fréquences propres : 66.78 Hz et 138.50 Hz
solution de l'équation différentielle : x_0*(alpha*sin(omega_d*t) + omega_d*cos(omega_d*t))*exp(-alpha*t)/omega_d
rapport de deux maxima successifs : exp(2*pi*alpha/omega_d)
décrément logarithmique en lettres : 2*pi*alpha/omega_d
m = 5 kg, k = 20 000 N/m, zeta = 0,05 : f0 = 10.07 Hz, décrément logarithmique = 0.3146
Exercice 10.2 : contrôles passés


## Exercice 10.3 : Fréquences propres et treillis (défi, 30 min)

*Première partie.* Reprenez l'équation caractéristique des deux masses de l'exercice 10.2 en gardant $k_2$ comme lettre. Résolvez-la pour $\lambda$, formez les deux fréquences $f = \sqrt{\lambda}/(2\pi)$, changez-les en une fonction avec `lambdify`, et calculez les deux fréquences pour $k_2 = 0{,}5$, $1$ et $2$ MN/m ($m_1 = 5$ kg, $m_2 = 3$ kg, $k_1 = 2$ MN/m).

*Seconde partie.* Un treillis de Warren isostatique compte cinq nœuds, de coordonnées en millimètres $0 : (0\,;\,0)$, $1 : (2\,000\,;\,0)$, $2 : (4\,000\,;\,0)$, $3 : (1\,000\,;\,1\,500)$, $4 : (3\,000\,;\,1\,500)$, et sept barres, dans l'ordre $0$-$1$, $1$-$2$, $3$-$4$, $0$-$3$, $3$-$1$, $1$-$4$, $4$-$2$. Il est articulé au nœud $0$ (deux réactions) et appuyé verticalement au nœud $2$ (une réaction) ; une charge de $20$ kN vers le bas s'applique au nœud $1$. Écrivez l'équilibre des cinq nœuds, soit dix équations, pour les sept efforts normaux (positifs en traction) et les trois réactions, et résolvez le système avec `linsolve`. Quelles barres sont comprimées ?

**Résultat attendu.** Pour $k_2 = 1$ MN/m, 66,78 Hz et 138,5 Hz, les deux fréquences croissant avec $k_2$ (figure du chapitre) ; efforts en kN, dans l'ordre des barres : 6,67, 6,67, $-13{,}33$, $-12{,}02$, 12,02, 12,02 et $-12{,}02$ ; trois barres comprimées.

**Seconde voie.** Par symétrie, chaque appui reprend la moitié de la charge ; la barre $0$-$3$ projette son effort sur la verticale avec le facteur $1\,500/\sqrt{1\,000^2 + 1\,500^2}$ et équilibre cette réaction, d'où un effort de même valeur absolue que celui de la barre $3$-$1$, 12,02 kN.

**Indication.** Première partie : `solve` sur le polynôme en $\lambda$ rend deux racines ; la racine carrée divisée par $2\pi$ donne les fréquences, que `lambdify` change en fonction de $(m_1, m_2, k_1, k_2)$. Seconde partie : pour chaque barre, la composante de l'effort sur chaque axe s'ajoute au nœud d'un bout et se retranche à l'autre ; les réactions et la charge complètent les équations. `linsolve` rend les sept efforts et les trois réactions ; les barres $3$-$4$, $0$-$3$ et $4$-$2$ sont comprimées.

In [6]:
# Solution :
fleche_np = sp.lambdify((F, L, E, b, h), fleche, "numpy")
hauteurs_mm = np.array([30.0, 40.0, 50.0, 60.0])
fleches_mm = fleche_np(2000.0, 1000.0, 210000.0, 20.0, hauteurs_mm)
print("flèche pour h =", hauteurs_mm, "mm :", np.round(fleches_mm, 2), "mm")
# Fin de la solution
assert abs(fleche_np(2000.0, 1000.0, 210000.0, 20.0, 40.0) - fleche_mm) < 1e-12
assert np.allclose(np.round(fleches_mm, 2), [4.41, 1.86, 0.95, 0.55])      # valeurs du chapitre 3
# Solution :
W = sp.Symbol("W", positive=True)                  # Kt de Heywood, avec d et W comme lettres
kt_sym = 2 + (1 - d / W)**3
kt_np = sp.lambdify((d, W), kt_sym, "numpy")
rapports = np.array([0.1, 0.2, 0.3, 0.4, 0.8])
print("Kt de Heywood pour d/W =", rapports, ":", np.round(kt_np(rapports * 50.0, 50.0), 3))
kt_035 = float(kt_np(0.35 * 50.0, 50.0))
kt_noeud_03, kt_noeud_04 = float(kt_np(0.3 * 50.0, 50.0)), float(kt_np(0.4 * 50.0, 50.0))
kt_035_lineaire = (kt_noeud_03 + kt_noeud_04) / 2          # lecture entre les deux nœuds voisins
print(f"Kt(0,35) exact : {kt_035:.4f}   lu entre d/W = 0,3 et 0,4 : {kt_035_lineaire:.4f}")
# Fin de la solution
assert abs(kt_035 - 2.2746) < 5e-5 and abs(kt_035_lineaire - 2.2795) < 5e-5
# Solution :
# Défi, première partie : les fréquences propres de deux masses, pour tout k2.
racines_lit = sp.solve(polynome, lam)               # deux racines en lettres
frequences_sym = [sp.sqrt(rac) / (2 * sp.pi) for rac in racines_lit]
frequences_np = sp.lambdify((m1, m2, k1, k2), frequences_sym, "numpy")
k2_scan = np.array([0.5e6, 1.0e6, 2.0e6])
f_bas, f_haut = sorted(frequences_np(5.0, 3.0, 2.0e6, k2_scan), key=lambda arr: arr[0])
print("k2 (N/m)           :", k2_scan)
print("première fréquence :", np.round(f_bas, 2), "Hz")
print("seconde fréquence  :", np.round(f_haut, 2), "Hz")
f1_nombre, f2_nombre = (float(f[1]) for f in (f_bas, f_haut))
# Fin de la solution
assert abs(f1_nombre - frequences_Hz[0]) < 1e-9 and abs(f2_nombre - frequences_Hz[1]) < 1e-9
# Solution :
# Défi, seconde partie : treillis de Warren isostatique, efforts dans les barres par l'équilibre des nœuds.
# Cinq nœuds, sept barres, trois liaisons (deux au nœud 0, une au nœud 2) : 10 équations pour 10 inconnues.
a_mm, h_mm = sp.Integer(2000), sp.Integer(1500)
noeuds = [(0, 0), (a_mm, 0), (2 * a_mm, 0), (a_mm / 2, h_mm), (3 * a_mm / 2, h_mm)]
barres = [(0, 1), (1, 2), (3, 4), (0, 3), (3, 1), (1, 4), (4, 2)]
charge_N = sp.Integer(20000)                         # charge vers le bas au nœud 1
N = sp.symbols("N1:8")                               # efforts normaux, positifs en traction
liaisons = sp.symbols("Rx0 Ry0 Ry2")
equations = [0] * 10
longueurs_mm = []

for (i, j), effort in zip(barres, N):
    dx, dy = noeuds[j][0] - noeuds[i][0], noeuds[j][1] - noeuds[i][1]
    longueur = sp.sqrt(dx**2 + dy**2)
    longueurs_mm.append(longueur)
    equations[2 * i] += effort * dx / longueur           # la barre tendue tire le nœud i vers le nœud j
    equations[2 * i + 1] += effort * dy / longueur
    equations[2 * j] -= effort * dx / longueur
    equations[2 * j + 1] -= effort * dy / longueur

equations[0] += liaisons[0]
equations[1] += liaisons[1]
equations[5] += liaisons[2]
equations[3] -= charge_N
solution_treillis = list(sp.linsolve(equations, list(N) + list(liaisons)))[0]
efforts_kN = [float(n / 1000) for n in solution_treillis[:7]]
print("efforts dans les barres (kN, + traction) :", [round(n, 2) for n in efforts_kN])
print("liaisons (kN) :", [round(float(s / 1000), 2) for s in solution_treillis[7:]])
# Travail des efforts : flèche du nœud 1 = somme de N**2 L / (F E A), pour E = 210 000 MPa et A = 300 mm2
E_MPa, A_mm2 = sp.Integer(210000), sp.Integer(300)
fleche_noeud_mm = sum(n**2 * lg for n, lg in zip(solution_treillis[:7], longueurs_mm)) / (charge_N * E_MPa * A_mm2)
fleche_treillis_mm = -float(fleche_noeud_mm)         # axe y vers le haut : le nœud descend
print(f"flèche du nœud 1 : {fleche_treillis_mm:.4f} mm")
intensites = [round(abs(n), 2) for n in efforts_kN]
# Fin de la solution
assert intensites == [6.67, 6.67, 13.33, 12.02, 12.02, 12.02, 12.02]
assert abs(sum(efforts_kN[:2]) / 2 - 6.6667) < 1e-3 and abs(solution_treillis[7]) < 1e-9
assert abs(fleche_treillis_mm + 1.25) < 0.005, "flèche du treillis attendue : -1,25 mm"
print("Exercice 10.3 : contrôles passés")

flèche pour h = [30. 40. 50. 60.] mm : [4.41 1.86 0.95 0.55] mm
Kt de Heywood pour d/W = [0.1 0.2 0.3 0.4 0.8] : [2.729 2.512 2.343 2.216 2.008]
Kt(0,35) exact : 2.2746   lu entre d/W = 0,3 et 0,4 : 2.2795
k2 (N/m)           : [ 500000. 1000000. 2000000.]
première fréquence : [55.71 66.78 73.39] Hz
seconde fréquence  : [117.4  138.5  178.23] Hz
efforts dans les barres (kN, + traction) : [6.67, 6.67, -13.33, -12.02, 12.02, 12.02, -12.02]
liaisons (kN) : [0.0, 10.0, 10.0]
flèche du nœud 1 : -1.2500 mm
Exercice 10.3 : contrôles passés


## Essais des fiches de fonction du chapitre

Chaque fiche du livre donne l'appel minimal d'une fonction. Les cellules suivantes rejouent ces appels, avec les mêmes lignes de code et la même sortie que la fiche : changez une valeur et relisez le résultat.

### Fiche F10-1 : Équations symboliques (`sympy.solve`)

In [7]:
import sympy as sp

t, N, Mt, r, se = sp.symbols("t N M_t r sigma_e", positive=True)
sigma = N / (2 * sp.pi * r * t)                 # contrainte normale
tau = Mt / (2 * sp.pi * r**2 * t)               # contrainte de cisaillement
equation = sp.Eq(sigma**2 + 3 * tau**2, se**2)  # critère de von Mises
solution = sp.solve(equation, t)[0]             # isoler l'épaisseur t

cas = {N: 50e3, Mt: 200, r: 0.02, se: 200e6}    # N, N.m, m, Pa
t_num = float(solution.subs(cas)) * 1e3         # conversion en mm
print(f"Épaisseur minimale : {t_num:.2f} mm".replace(".", ","))
controle = sp.simplify(equation.lhs.subs(t, solution) - se**2)
print(f"Équilibre exact : {controle == 0}")

Épaisseur minimale : 2,11 mm
Équilibre exact : True


### Fiche F10-2 : Du symbole au tableau (`sympy.lambdify`)

In [8]:
import numpy as np
import sympy as sp

T, L, G, D, d = sp.symbols("T L G D d", positive=True)
theta = 32 * T * L / (sp.pi * G * (D**4 - d**4))          # angle de torsion
f_theta = sp.lambdify((T, L, G, D, d), theta, "numpy")
d_int = np.array([0.02, 0.03, 0.04])                      # alésages, m
angles = f_theta(500.0, 1.2, 80e9, 0.05, d_int)           # en radians
print(" ".join(f"{x:.3f}" for x in angles * 180 / np.pi).replace(".", ","))
J_np = np.pi * (0.05**4 - d_int**4) / 32
print(np.allclose(angles, 500.0 * 1.2 / (80e9 * J_np)))   # contrôle

0,719 0,805 1,186
True
